Eco friendly agents for Q4. Each agent is a small simulation. Every action uses some energy and that energy is converted into CO2 and water. Assumed values are 0.45 g CO2 per Wh and 1.8 ml water per Wh and 250 g CO2 per km for a diesel van.

In [1]:
import random, math, numpy as np
random.seed(1); np.random.seed(1)
CO2_PER_WH=0.45; WATER_PER_WH=1.8; VAN_CO2_PER_KM=250
def footprint(wh): return wh*CO2_PER_WH, wh*WATER_PER_WH

Agent A: Reflex agent for abnormal movement in a crowd. It reads the crowd speed and raises an alert if the speed is above 3 m/s or below 0.05 m/s. A low power motion sensor stays on and the camera wakes up only when movement is abnormal. The normal agent keeps the camera on all the time so the eco agent gives the same alerts with much less energy.

In [2]:
class CrowdReflexAgent:
    def __init__(self, eco=True): self.eco=eco; self.wh=0; self.alerts=0
    def act(self, speed):
        abnormal = speed>3.0 or speed<0.05
        if self.eco:
            self.wh += 0.5/60
            if abnormal: self.wh += 5/60; self.alerts+=1; return 'ALERT'
            return 'idle'
        self.wh += 5/60
        if abnormal: self.alerts+=1; return 'ALERT'
        return 'idle'
speeds=np.concatenate([np.random.normal(1.2,0.3,550),np.random.normal(4,0.5,10)]); np.random.shuffle(speeds)
for eco in (False,True):
    a=CrowdReflexAgent(eco); [a.act(s) for s in speeds]
    print('eco' if eco else 'normal', 'alerts:',a.alerts,'energy Wh: %.2f'%a.wh,'CO2 g: %.2f'%footprint(a.wh)[0],'water ml: %.2f'%footprint(a.wh)[1])

normal alerts: 10 energy Wh: 46.67 CO2 g: 21.00 water ml: 84.00
eco alerts: 10 energy Wh: 5.50 CO2 g: 2.48 water ml: 9.90


Agent B: Utility based agent for stressed patients. Every action has a stress reduction and an energy cost. Utility is stress reduction minus 0.08 times energy and the agent picks the highest utility. For high stress it still calls the nurse so care is not reduced.

In [3]:
ACTIONS={
 'do_nothing':(0,0),'dim_lights':(2,1),'play_calm_music':(3,2),'guided_breathing_app':(4,3),
 'aircon_cooling':(3,40),'call_nurse':(8,6)}
def utility(a,stress,w_eco=0.08):
    red,wh=ACTIONS[a]; return min(red,stress)-w_eco*wh
def choose(stress,eco=True):
    if not eco: return max(ACTIONS,key=lambda a:ACTIONS[a][0])
    return max(ACTIONS,key=lambda a:utility(a,stress))
tot={True:0,False:0}
for s in [2,4,6,9,3,7,8,1]:
    for eco in (True,False):
        a=choose(s,eco); tot[eco]+=ACTIONS[a][1]
    print('stress',s,'| eco picks',choose(s,True),'| normal picks',choose(s,False))
for eco in (True,False): print('eco' if eco else 'normal','Wh:',tot[eco],'CO2 g: %.1f'%footprint(tot[eco])[0])

stress 2 | eco picks dim_lights | normal picks call_nurse
stress 4 | eco picks guided_breathing_app | normal picks call_nurse
stress 6 | eco picks call_nurse | normal picks call_nurse
stress 9 | eco picks call_nurse | normal picks call_nurse
stress 3 | eco picks play_calm_music | normal picks call_nurse
stress 7 | eco picks call_nurse | normal picks call_nurse
stress 8 | eco picks call_nurse | normal picks call_nurse
stress 1 | eco picks dim_lights | normal picks call_nurse
eco Wh: 31 CO2 g: 14.0
normal Wh: 48 CO2 g: 21.6


Agent C: Goal based agent for the school van network. The goal is to pick up all students and reach school by a safe route with the least distance. It checks every stop order and drops the unsafe ones and keeps the shortest. Less distance means less fuel and less CO2.

In [4]:
import itertools
pts={'School':(0,0),'S1':(2,1),'S2':(3,4),'S3':(-2,3),'S4':(-3,-2)}
def d(a,b): return math.dist(pts[a],pts[b])
unsafe={('S1','S4'),('S4','S1')}
def route_len(order):
    r=['School']+list(order)+['School']; L=0
    for a,b in zip(r,r[1:]):
        if (a,b) in unsafe: return None
        L+=d(a,b)
    return L
stops=['S1','S2','S3','S4']
valid={o:route_len(o) for o in itertools.permutations(stops) if route_len(o)}
best=min(valid,key=valid.get); naive=('S2','S4','S3','S1')
for name,o in (('naive',naive),('goal-based best',best)):
    L=route_len(o) or float('nan'); print(name,o,'km: %.2f'%L,'CO2 kg: %.2f'%(L*VAN_CO2_PER_KM/1000))
print('Saved CO2 (g):', round((route_len(naive)-valid[best])*VAN_CO2_PER_KM,1) if route_len(naive) else 'naive route is unsafe')

naive ('S2', 'S4', 'S3', 'S1') km: 25.29 CO2 kg: 6.32
goal-based best ('S1', 'S2', 'S3', 'S4') km: 19.20 CO2 kg: 4.80
Saved CO2 (g): 1522.6


Agent D: Learning agent to encourage sports in young people. It learns which activity to suggest using epsilon greedy learning. The reward is 1 if the person does the activity minus a penalty for car CO2. With time it suggests walking and cycling more and the car trip less.

In [5]:
acts=['drive_to_gym','cycle_to_park','walk_with_friends','home_workout']
engage={'drive_to_gym':0.55,'cycle_to_park':0.65,'walk_with_friends':0.6,'home_workout':0.5}
co2={'drive_to_gym':800,'cycle_to_park':0,'walk_with_friends':0,'home_workout':5}
Qv={a:0 for a in acts}; N={a:0 for a in acts}; hist=[]; tot_co2=0
for t in range(500):
    a=random.choice(acts) if random.random()<0.1 else max(Qv,key=Qv.get)
    did=random.random()<engage[a]
    r=(1 if did else 0)-co2[a]/1000
    N[a]+=1; Qv[a]+=(r-Qv[a])/N[a]; tot_co2+=co2[a]*did; hist.append(a)
print({a:round(v,3) for a,v in Qv.items()}); print('chosen counts',N); print('total CO2 g:',tot_co2)
rand_co2=sum(co2[random.choice(acts)]*(random.random()<0.6) for _ in range(500)); print('random agent CO2 g ~',rand_co2)

{'drive_to_gym': -0.527, 'cycle_to_park': 0.537, 'walk_with_friends': 0.607, 'home_workout': 0.495}
chosen counts {'drive_to_gym': 11, 'cycle_to_park': 54, 'walk_with_friends': 425, 'home_workout': 10}
total CO2 g: 2425
random agent CO2 g ~ 61210


Result: all four eco agents use less energy or produce less CO2 than the normal version.